In [ ]:
import os
import time
import torch
import skimage
import sklearn.metrics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
import mnds
import extractor
import detection
import vision_transformer as vit

In [ ]:
PATCH_SIZE = 256
STRIDE = 8
FEATURE_SIZE = 384
TOKENS_PER_PATCH = PATCH_SIZE // STRIDE
STEP = 16

DIRECTORY = "/home/caicedo/scr/jcaicedo/Micronuclei-data/"

BATCH_SIZE = 512
EPOCHS = 20
LR = 0.01

device = 'cuda:2' if torch.cuda.is_available() else 'cpu'

In [ ]:
filelist = os.listdir(DIRECTORY)
annot_files = [x for x in filelist if x.endswith('png')]

validation_file = annot_files[9]
imid = validation_file.split('.')[0]

print(imid)

In [ ]:
model_file = DIRECTORY + "models/" + validation_file.replace('phenotype_outlines.png','pth')
model = torch.load(model_file)
model

In [ ]:
im = mnds.read_image(DIRECTORY, imid, 'phenotype.tif')
im = skimage.exposure.rescale_intensity(im, out_range=np.float32)

In [ ]:
probabilities = np.zeros((im.shape[0]//STRIDE, im.shape[1]//STRIDE), dtype=np.float32)
counts = np.zeros((im.shape[0]//STRIDE, im.shape[1]//STRIDE), dtype=np.float32)
ones = np.ones((TOKENS_PER_PATCH, TOKENS_PER_PATCH))
batch, coords = [], []

model.eval()

def predict(batch, coords):
    B = torch.cat(batch, axis=0)
    pred0 = model(B.to(device))
    P = torch.reshape(pred0, (-1, TOKENS_PER_PATCH, TOKENS_PER_PATCH))
    P = P.cpu().numpy()

    for c in range(len(coords)):
        y = coords[c]["a"]
        x = coords[c]["b"]
        probabilities[y:y+TOKENS_PER_PATCH,x:x+TOKENS_PER_PATCH] += P[c]
        counts[y:y+TOKENS_PER_PATCH,x:x+TOKENS_PER_PATCH] += ones
    coords = []

    
with torch.no_grad():
    for i in range(0,im.shape[0]-PATCH_SIZE+1, STEP):
        a = i // STRIDE
        for j in range(0,im.shape[1]-PATCH_SIZE+1, STEP):
            b = j // STRIDE
            vin = mnds.patch_to_rgb(im[i:i+PATCH_SIZE,j:j+PATCH_SIZE])
            batch.append(vin[None,:,:,:])
            coords.append({"i":i, "j":j, "a":a, "b":b})
            
            if len(batch) == BATCH_SIZE:
                # Get predictions
                predict(batch, coords)
                batch, coords = [], []
                
    if len(batch) > 0:
        predict(batch, coords)
        batch, coords = [], []

probabilities = probabilities/counts

In [ ]:
gt = mnds.read_micronuclei_annotations(DIRECTORY, imid)

ground_truth = np.zeros_like(probabilities)
for k,r in gt.iterrows():
    a = r.y // 8
    b = r.x // 8
    ground_truth[a,b] = 1

predictions = probabilities > 0.2

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(18,6))

ax[0].imshow(im)
ax[1].imshow(ground_truth)
ax[2].imshow(predictions)

In [ ]:
# Precision-recall curve

GT = ground_truth.flatten()
PRED = probabilities.flatten()

display = sklearn.metrics.PrecisionRecallDisplay.from_predictions(
    GT, PRED, name="Detector", plot_chance_level=True
)
_ = display.ax_.set_title("Precision-Recall curve")

In [ ]:
# Classification report
report = sklearn.metrics.classification_report(GT, PRED > 0.2)
print(report)

In [ ]:
correct = predictions * ground_truth
missing = ground_truth - correct
extra = predictions - correct
print("Total:",np.sum(ground_truth),"Correct:",np.sum(correct), "Missing:", np.sum(missing), "Extra:", np.sum(extra))

In [ ]:
import matplotlib.patches as patches
# Show image
fig, ax = plt.subplots(figsize=(30,30))
ax.imshow(im)

annotations = []

# Display micronucleus boxes
C = np.where(correct)
w,h = 16,16
for i in range(len(C[0])):
    x1 = C[1][i]*8 - w
    y1 = C[0][i]*8 - h
    rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=1, edgecolor='gold', facecolor='none')
    ax.add_patch(rect)
    
# Display micronucleus boxes
C = np.where(missing)
w,h = 12,12
for i in range(len(C[0])):
    x1 = C[1][i]*8 - w
    y1 = C[0][i]*8 - h
    rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=1, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    plt.text(x1, y1, i, color="r", fontsize="xx-large")
    annotations.append({"col":C[1][i]*8, "row":C[0][i]*8, "ID":i, "color":"red","question":"Missed?"})
    
# Display micronucleus boxes
C = np.where(extra)
w,h = 12,12
for i in range(len(C[0])):
    x1 = C[1][i]*8 - w
    y1 = C[0][i]*8 - h
    rect = patches.Rectangle((x1, y1), 2*w, 2*h, linewidth=1, edgecolor='b', facecolor='none')
    ax.add_patch(rect)
    plt.text(x1, y1, i, color="b", fontsize="xx-large")
    annotations.append({"col":C[1][i]*8, "row":C[0][i]*8, "ID":i, "color":"blue","question":"Real?"})

#plt.axis('off')
#plt.show()
plt.savefig(f"{DIRECTORY}/predictions/{imid}-fig.png")

In [ ]:
df = pd.DataFrame(annotations)
df["answer"] = ""
df = df.sort_values(by=["ID","question"])
df.to_csv(f"{DIRECTORY}/predictions/{imid}-checks.csv")